In [ ]:
from math import exp
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels as sm
import statsmodels.formula.api as smf
from statsmodels.stats import outliers_influence
from statsmodels.discrete.count_model import ZeroInflatedPoisson, ZeroInflatedNegativeBinomialP
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.factorplots import interaction_plot
import seaborn as sns
import scipy
import patsy
from IPython.display import display, HTML
sns.set_theme()

In [ ]:
filepath = os.getcwd()

# Import crowdsourced data
Waze_reports = pd.read_csv(f'{filepath}/Waze_reports.csv')
road_miles = pd.read_csv(f'{filepath}/road_miles_categorized.csv')
print(len(Waze_reports))

Waze_reports = Waze_reports.merge(road_miles[['id', 'mmi_bin', 'highway_meters', 'non_highway_meters']], on=['id', 'mmi_bin'])
print(len(Waze_reports))

# Drop earthquakes from the beginning of the study period with no control period
Waze_reports = Waze_reports[~Waze_reports['id'].isin(['nc72766046', 'nc72771146', 'us6000fr7b', 'uw61251926'])]

# Import police-reported data (for later comparison)
CHP_reports = pd.read_csv(f'{filepath}/CHP_reports.csv')
road_miles_ca = pd.read_csv(f'{filepath}/road_miles_categorized_CA.csv')

CHP_reports = CHP_reports.merge(road_miles_ca[['id', 'mmi_bin', 'highway_meters', 'non_highway_meters']], on=['id', 'mmi_bin'])

In [ ]:
spotchecking = Waze_reports[Waze_reports['alert_time_bin'].str.contains('Control')].groupby(['id', 'time', 'mmi_bin', 'alert_time_bin'], as_index=False).agg({'week_num': ['count', 'min']})

spotchecking

In [ ]:
# Round and format crowdsourced data
Waze_rounded = Waze_reports.copy(deep=True)
Waze_rounded['mmi_bin'] = np.round(Waze_rounded['mmi_bin'], 1)
Waze_rounded = Waze_rounded.groupby(['id', 'mmi_bin', 'time', 'week_num','alert_time_bin'], as_index=False).sum()
Waze_rounded['total_ThouKMs'] = (Waze_rounded['length_meters']/1000)/1000
Waze_rounded['highway_ThouKMs'] = (Waze_rounded['highway_meters']/1000)/1000
Waze_rounded['non_highway_ThouKMs'] = (Waze_rounded['non_highway_meters']/1000)/1000

# Round and format police-reported data
CHP_rounded = CHP_reports.copy(deep=True)
CHP_rounded['mmi_bin'] = np.round(CHP_rounded['mmi_bin'], 1)
CHP_rounded = CHP_rounded.groupby(['id', 'mmi_bin', 'time', 'week_num','alert_time_bin'], as_index=False, ).sum()
CHP_rounded['total_ThouKMs'] = (CHP_rounded['length_meters']/1000)/1000
CHP_rounded['highway_ThouKMs'] = (CHP_rounded['highway_meters']/1000)/1000
CHP_rounded['non_highway_ThouKMs'] = (CHP_rounded['non_highway_meters']/1000)/1000

bincols = ['1hrControl', '1hrAfter', '2hrControl', '2hrAfter', '12hrControl', '12hrAfter', '7dayControl', '7dayAfter']
binDurations = [1, 1, 2, 2, 12, 12, 168, 168]

In [ ]:
areas = pd.read_csv(f'{filepath}\Area_By_EQ.csv')
print(len(Waze_rounded))
Waze_rounded = Waze_rounded.merge(areas, on=['id', 'mmi_bin'])
print(len(Waze_rounded))
Waze_rounded

In [ ]:
# Create flags for control periods and post-earthquake periods
Waze_rounded['Control'] = [1 if 'Control' in tb else 0 for tb in Waze_rounded.alert_time_bin]
Waze_rounded['After'] = [1 if 'After' in tb else 0 for tb in Waze_rounded.alert_time_bin]
def hoursAfter(tbin):
    if '12hr' in tbin:
        return '12hr'
    if '1hr' in tbin:
        return '1hr'
    if '2hr' in tbin:
        return '2hr'
    if '7day' in tbin:
        return '7day'

Waze_rounded['hoursAfter'] = Waze_rounded.alert_time_bin.apply(hoursAfter)

CHP_rounded['Control'] = [1 if 'Control' in tb else 0 for tb in CHP_rounded.alert_time_bin]
CHP_rounded['After'] = [1 if 'After' in tb else 0 for tb in CHP_rounded.alert_time_bin]
def hoursAfter(tbin):
    if '12hr' in tbin:
        return '12hr'
    if '1hr' in tbin:
        return '1hr'
    if '2hr' in tbin:
        return '2hr'
    if '7day' in tbin:
        return '7day'

CHP_rounded['hoursAfter'] = CHP_rounded.alert_time_bin.apply(hoursAfter)

Waze_rounded

In [ ]:

countsEQ0 = Waze_rounded[Waze_rounded['After']==0].groupby(['id', 'After', 'hoursAfter', 'week_num'], as_index=False)['num_crashes'].sum()
countsEQ1 = Waze_rounded[Waze_rounded['After']==1].groupby(['id', 'After', 'hoursAfter', 'week_num'], as_index=False)['num_crashes'].sum()
countsEQ0 = countsEQ0.groupby(['id', 'After', 'hoursAfter'], as_index=False)['num_crashes'].mean()
countsEQ1 = countsEQ1.groupby(['id', 'After', 'hoursAfter'], as_index=False)['num_crashes'].mean()

countsEQ = countsEQ0.merge(countsEQ1, on=['id', 'hoursAfter'], suffixes = ['_Control', '_Earthquake'])

countsEQ

In [ ]:
# Re-aggregate to the time bins to be used for the run. Select the desired time bin and all smaller time bins. For example, to model the 2hr time bin, select both the 1hr and 2hr time bins.
# tbins = ['1hr'] # 1hr time bin
tbins = ['1hr', '2hr'] # 2hr time bin
# tbins = ['1hr', '2hr', '12hr'] # 12hr time bin
# tbins = ['1hr', '2hr', '12hr', '7day'] # 7day time bin
df = Waze_rounded[Waze_rounded['hoursAfter'].isin(tbins)].groupby(['id', 'mmi_bin', 'time', 'week_num', 'total_ThouKMs', 'highway_ThouKMs', 'non_highway_ThouKMs', 'After', 'area_sqkm'], as_index=False)[['num_crashes']].sum()

# Drop the Ridgecrest earthquake sequence
df = df[~df['id'].isin(["ci38443183", "ci38445703", "ci38457511", "ci38465703", "ci38469375"])]

df = df[df['num_crashes']>=0]
df['total_ThouKMs_lin'] = np.log10(df['total_ThouKMs'])
df = sm.tools.tools.add_constant(df)

print(df.groupby(['After'])['num_crashes'].describe())
print(df[df['num_crashes']>0].groupby(['After'])['num_crashes'].describe())

In [ ]:
# Calculate Poisson means test results for MMI bins
test = df
for m in np.arange(2, 8, 0.5):

    print("---------------------")
    print(f"**** poisson means test results for mmi at least {m} ****")
    kc = test[(test['After']==0)&(test['mmi_bin']>=m)].num_crashes.sum()
    nc = test[(test['After']==0)].num_crashes.sum()
    ka = test[(test['After']==1)&(test['mmi_bin']>=m)].num_crashes.sum()
    na = test[(test['After']==1)].num_crashes.sum()
    print(f'{kc}/{nc} = {kc/nc}, {ka}/{na}={ka/na}')
    print(scipy.stats.poisson_means_test(kc, nc, ka, na, alternative='two-sided'))
    print("---------------------")

In [ ]:
models = {}

# Gaussian
df['crashes_per_ThouKMs'] = df['num_crashes']/df['total_ThouKMs']
form = 'crashes_per_ThouKMs ~ mmi_bin*C(After)'

model = smf.glm(formula = form, data = df)
results = model.fit(maxiter=60)
models['model1'] = {'name': 'GLM, with interaction term', 'form': form, 'model': results}

display(HTML(results.summary().as_html()))
print(results.pvalues)
print(f'-- AIC: {results.aic}')

In [ ]:
# Poisson
form = 'num_crashes ~ mmi_bin*C(After) + total_ThouKMs'

model = smf.glm(formula = form, data = df, family=sm.genmod.families.family.Poisson())

results = model.fit(maxiter=60)
models['model2'] = {'name': 'Poisson, with interaction term', 'form': form, 'model': results}

display(HTML(results.summary().as_html()))
print(results.pvalues)
print(f'-- AIC: {results.aic}')

In [ ]:
# Negative Binomial
form = 'num_crashes ~ mmi_bin*C(After) + total_ThouKMs'

model = smf.glm(formula = form, data = df, family=sm.genmod.families.family.NegativeBinomial())

results = model.fit(maxiter=60)
models['model3'] = {'name': 'NBinomial, with interaction term', 'form': form, 'model': results}

display(HTML(results.summary().as_html()))
print(results.pvalues)
print(f'-- AIC: {results.aic}')

In [ ]:
# Zero-Inflated Poisson
form = 'num_crashes ~ mmi_bin*C(After) + highway_ThouKMs + non_highway_ThouKMs'
model = sm.discrete.count_model.ZeroInflatedPoisson.from_formula(formula = form, data = df)

results = model.fit_regularized(maxiter=150)
models['model5'] = {'name': 'Zero Inflated Poisson, with interaction term', 'form': form, 'model': results}

display(HTML(results.summary().as_html()))
print(results.pvalues)
print(f'-- AIC: {results.aic}')

In [ ]:
# Zero-Inflated Negative Binomial
df['hw_ratio'] = df['highway_ThouKMs']/df['total_ThouKMs']
df['nhw_ratio'] = df['non_highway_ThouKMs']/df['total_ThouKMs']
df['area_ThouSqKM'] = df['area_sqkm']/1000

form = 'num_crashes ~ mmi_bin*C(After) + highway_ThouKMs + non_highway_ThouKMs'
model = sm.discrete.count_model.ZeroInflatedNegativeBinomialP.from_formula(formula = form, data = df)

results = model.fit_regularized(maxiter=150)
models['model7'] = {'name': 'Zero Inflated NBinom, with interaction term', 'form': form, 'model': results}

display(HTML(results.summary().as_html()))
print(results.pvalues)
print(f'-- AIC: {results.aic}')

In [ ]:
# Compare the AIC for each model to find the model of best fit
aicList = np.array([])
for k, m in zip(models.keys(), models.values()):
    print('##-------------------##')
    print(f"{k}, {m['name']}")
    print(m['model'].aic)
    if hasattr(m['model'], 'pseudo_rsquared') and callable(getattr(m['model'], 'pseudo_rsquared')):
        print(m['model'].pseudo_rsquared('mcf'))
    if hasattr(m['model'], 'prsquared'):
        print(m['model'].prsquared)
    aicList = np.append(aicList, m['model'].aic)

print('##-------------------##')
aicMin = aicList.min()
print(f'AIC Min = {aicMin}')
for aic in aicList:
    print(exp((aicMin - aic)/2.0))

for k, m in zip(models.keys(), models.values()):
    if m['model'].aic == aicMin:
        bestmodel = k

In [ ]:
# Add constant term for intercept
y, X = patsy.dmatrices('num_crashes ~ mmi_bin*C(After) + highway_ThouKMs + non_highway_ThouKMs', data=df, return_type='dataframe')

# Check matrix rank: if rank < number of columns, there is perfect multicollinearity
rank = np.linalg.matrix_rank(X)
n_cols = X.shape[1]

print(f"Matrix rank: {rank}")
print(f"Number of columns: {n_cols}")

if rank < n_cols:
    print("Perfect multicollinearity detected!")
else:
    print("No perfect multicollinearity.")

In [ ]:
# Plot the average crashes per observation period for each MMI bin
timebins = {
    #'1 Hour': ['1hr'] 
    '2 Hour': ['1hr', '2hr'] 
    #'12 Hour': ['1hr', '2hr', '12hr']
    #'7 Days': ['1hr', '2hr', '12hr', '7day']
    }
fig, ax = plt.subplots(1, 1, sharex=True, sharey=False)
ax = [[ax,],]
fig.set_figheight(5)
fig.set_figwidth(8)
ks = [key for key in timebins.keys()]
print(ks)
k = 0
for i in ax:
    for j in i:
        tp = Waze_rounded[Waze_rounded['hoursAfter'].isin(timebins[ks[k]])].groupby(['id', 'mmi_bin', 'time', 'week_num', 'total_ThouKMs', 'After'], as_index=False).num_crashes.sum()
        tp['mmi_round'] = round(tp['mmi_bin']*2)/2
        sm.graphics.factorplots.interaction_plot(round(tp['mmi_bin']*2)/2, ['Control' if a == 0 else 'Earthquake' for a in tp['After']], tp['num_crashes'], colors=['blue','red'], ax = j, legendtitle='Time Period')
        j.set_title(ks[k])
        j.set_ylabel('Average Crashes per Observation Period')
        j.set_xlabel('MMI in Region')
        print(tp.groupby(['mmi_round', 'After'])['num_crashes'].sum())
        print(tp.groupby(['After'])['num_crashes'].sum())
        print(tp.groupby(['After'])['num_crashes'].count())
        k+=1

In [ ]:
# Plot the crashes per time bin at each MMI value alongside the roadway length at those MMI values
timebins = {
    '1 Hour': ['1hr'],
    '2 Hour': ['1hr', '2hr'],
    '12 Hour': ['1hr', '2hr', '12hr'],
    '7 Days': ['1hr', '2hr', '12hr', '7day']
    }
fig, ax = plt.subplots(3, 1, sharex=True, sharey=False)
fig.set_figheight(8)
fig.set_figwidth(10)
ks = [key for key in timebins.keys()]
print(ks)
k = 1

j = ax[0]
Waze_filtered = Waze_rounded[~Waze_rounded['id'].isin(["ci38443183", "ci38445703", "ci38457511", "ci38465703", "ci38469375"])]

tpW = Waze_filtered[Waze_filtered['hoursAfter'].isin(timebins[ks[k]])].groupby(['id', 'mmi_bin', 'time', 'week_num', 'total_ThouKMs', 'After'], as_index=False).num_crashes.sum()
sm.graphics.factorplots.interaction_plot(round(tpW['mmi_bin']*2)/2, ['Control' if a == 0 else 'Earthquake' for a in tpW['After']], tpW['num_crashes'], colors=['blue','red'], ax = j, legendtitle='Time Period')

means = tpW[tpW['After']==0].groupby('mmi_bin')['num_crashes'].mean()
sums = tpW[tpW['After']==0].groupby('mmi_bin')['num_crashes'].sum()

conf =  [scipy.stats.poisson.interval(confidence=.90, mu=m, loc=m) for m, s in zip(means, sums)]
conf = pd.DataFrame(columns=['lower', 'upper'], data=conf)
conf['mmi'] = tpW['mmi_bin'].unique()

j.set_ylabel('Waze Crashes per\n2-hour Observation Period')
j.set_xlabel('MMI Value')
j.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{x:.2f}'))


j = ax[1]
CHP_filtered = CHP_rounded[~CHP_rounded['id'].isin(["ci38443183", "ci38445703", "ci38457511", "ci38465703", "ci38469375"])]

tpC = CHP_filtered[CHP_filtered['hoursAfter'].isin(timebins[ks[k]])].groupby(['id', 'mmi_bin', 'time', 'week_num', 'total_ThouKMs', 'After'], as_index=False).num_crashes.sum()
tpC['mmi_round'] = round(tpC['mmi_bin']*2)/2
sm.graphics.factorplots.interaction_plot(round(tpC['mmi_bin']*2)/2, ['Control' if a == 0 else 'Earthquake' for a in tpC['After']], tpC['num_crashes'], colors=['blue','red'], ax = j, legendtitle='Time Period')

j.set_ylabel('CHP Crashes per\n2-hour Observation Period')
j.set_xlabel('MMI in Region')
j.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: f'{x:.2f}'))

totalCounts = Waze_rounded[(Waze_rounded['hoursAfter'].isin(timebins['2 Hour']))&
                           (Waze_rounded['After']==1)].groupby('mmi_bin', as_index=False)[['total_ThouKMs', 'highway_ThouKMs','non_highway_ThouKMs', 'num_crashes']].sum()
totalCounts['mmi_bin'] = round(2*totalCounts['mmi_bin'], 0)/2
totalCounts = totalCounts.groupby('mmi_bin', as_index=False)[['total_ThouKMs', 'highway_ThouKMs','non_highway_ThouKMs', 'num_crashes']].sum()
print(totalCounts)
totalCounts['total_ThouKMs'] = totalCounts['total_ThouKMs']/1000
totalCounts['highway_ThouKMs'] = totalCounts['highway_ThouKMs']/1000
totalCounts['non_highway_ThouKMs'] = totalCounts['non_highway_ThouKMs']/1000
totalCounts.plot(x='mmi_bin', y='highway_ThouKMs', ax = ax[2], style='.-', color='green', label = 'Total Highway Kilometers')
totalCounts.plot(x='mmi_bin', y='non_highway_ThouKMs', ax = ax[2], style='.-', color='purple', label = 'Total Non-Highway Kilometers')

ax[2].set_xlabel('MMI in Regions or on Roadways (All Earthquakes)')
ax[2].set_ylabel('Road Kilometers\n(Millions of KMs)')
plt.tight_layout()
plt.show()

print(totalCounts[totalCounts['mmi_bin']>=3.5][['total_ThouKMs', 'highway_ThouKMs', 'non_highway_ThouKMs']].sum())
print(totalCounts[totalCounts['mmi_bin']>=3.5]['highway_ThouKMs'].sum()/totalCounts[totalCounts['mmi_bin']>=3.5]['total_ThouKMs'].sum())
print(totalCounts[totalCounts['mmi_bin']>=3.5]['non_highway_ThouKMs'].sum()/totalCounts[totalCounts['mmi_bin']>=3.5]['total_ThouKMs'].sum())